In [1]:
# Instalar o pacote do ultralytics
!pip install ultralytics

In [2]:
# Instalar o pacote da openAI
!pip install -U openai

In [3]:
# Instalar o pacote reportlab
!pip install -U reportlab

In [4]:
# Instalar o pacote fonts-dejavu (suporte para caracteres unicode)
!apt-get update -y
!apt-get install -y fonts-dejavu

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  fonts-dejavu-core fonts-dejavu-extra
Th

In [5]:
# Importar bibliotecas
from ultralytics import YOLO
from pathlib import Path
from collections import Counter
import pandas as pd
import sys
import os
import json
import openai
from google.colab import userdata
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import cm
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, PageBreak
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_LEFT
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont

In [6]:
# Montar o drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
# Configurar o path até os scripts
sys.path.insert(0, '/content/drive/MyDrive/Tech Challenge 5/scripts')

In [8]:
# Criar uma variavel de ambiente que armazene a chave do segredo
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [9]:
# Importar as bibliotecas do script detectar_componentes_com_yolo.py
from detectar_componentes_com_yolo import gerar_lista_imagens, carregar_yolo, aplicar_yolo, gerar_dataframe, gerar_relatorio_csv

# Importar as bibliotecas do script classificar_componentes_detectados.py
from classificar_componentes_detectados import gerar_relatorio_classes

# Importar as bibliotecas do script retornar_arquivos_rag.py
from retornar_arquivos_rag import retornar_documento_rag

# Importar as bibliotecas do script funcoes_llm.py
from funcoes_llm import retornar_llm_prompts, enviar_pergunta_llm

# Importar as bibliotecas do script gerar_relatorios_stride.py
from gerar_relatorios_stride import gerar_relatorio_pdf

In [10]:
# Gerar a lista de imagens
images_list = gerar_lista_imagens(dir_path='/content/drive/MyDrive/Tech Challenge 5/imagens/imagens_source')

['arquitetura-aws.png', 'arquitetura-azure.png']


In [11]:
# Carregar o modelo yolo com o best.pt
yolo_model = carregar_yolo(model_path='/content/drive/MyDrive/Tech Challenge 5/resultados_yolo/fine_tuning_train/weights/best.pt')

Modelo yolo carregado.


In [12]:
# Aplicar yolo nas imagens listadas
yolo_results = aplicar_yolo(
    yolo_model=yolo_model,
    images_list=images_list,
    source='/content/drive/MyDrive/Tech Challenge 5/imagens/imagens_source',
    imgsz=960,
    conf=0.25,
    iou=0.5,
    save=True,
    name='/content/drive/MyDrive/Tech Challenge 5/imagens',
    verbose=False)


Detectando os componentes da imagem arquitetura-aws.png
Results saved to /content/drive/MyDrive/Tech Challenge 5/imagens/imagens_yolo_arquitetura-aws

Detectando os componentes da imagem arquitetura-azure.png
Results saved to /content/drive/MyDrive/Tech Challenge 5/imagens/imagens_yolo_arquitetura-azure

Imagens analisadas pelo yolo.


In [13]:
# Gerar o dataFrame com as detecções do yolo
df_results = gerar_dataframe(detections_list=yolo_results)


DataFrame gerado.


In [14]:
# Gerar o relatório csv com as detecções do yolo
df_report = gerar_relatorio_csv(dataframe=df_results, report_path='/content/drive/MyDrive/Tech Challenge 5/reports')


Relatório csv gerado com sucesso.


In [15]:
# Gerar o relatório csv com as classes dos componentes detectados
df_report_class = gerar_relatorio_classes(
    report_detections_path='/content/drive/MyDrive/Tech Challenge 5/reports/yolo_detections.csv',
    report_classes_path='/content/drive/MyDrive/Tech Challenge 5/reports/component_classes.xlsx',
    csv_output_path='/content/drive/MyDrive/Tech Challenge 5/reports/yolo_detections_classified.csv'
    )

Relatório csv com as classes dos componentes gerado com sucesso.


In [16]:
# Bloco principal onde as requisições ao llm são feitas e o relatório STRIDE é gerado.
for image in images_list:

  # Retornar um dataFrame com apenas as linhas da imagem selecionada
  df_report_class_filtered = df_report_class[df_report_class['nome_imagem'] == image]
  # Criar uma lista distinta com as classes detectadas
  distinct_class_list = df_report_class_filtered['classe_componente'].unique()

  # Variavel que irá armazenar as respostas da llm
  llm_respostas = []

  # Para cada classe na lista deve-se retornar o seu respectivo documento RAG e os seus componentes cloud em uma lista
  for item in distinct_class_list:

    # Inserir os parametros da llm [model, temperature, max_tokens]
    llm_parameters = ["gpt-5-mini", 1, 3000]

    # Receber o nome da classe
    class_name = item

    # Gerar lista de componentes da classe e transforma-la em json para ser passado como parâmetro
    df_class = df_report_class_filtered[df_report_class_filtered["classe_componente"] == class_name]
    components_list = df_class["nome_componente"].drop_duplicates().tolist()
    components_list_json = json.dumps(components_list, ensure_ascii=False, indent=2)

    # Retornar o documento RAG da classe
    json_file = retornar_documento_rag(
        document_path='/content/drive/MyDrive/Tech Challenge 5/rag_documents/',
        classe_name=class_name
        )

    # Retornar o prompt que será utilizado para a llm
    system_prompt, user_prompt = retornar_llm_prompts(class_name, components_list_json, json_file)

    # Maximo de tentativas para receber uma resposta válida do llm
    max_tentativas = 3

    # Retornar a resposta da llm
    for _ in range(max_tentativas):
      resposta = enviar_pergunta_llm(system_prompt, user_prompt, *llm_parameters)
      try:
        json.loads(resposta)
        llm_respostas.append(resposta)
        break
      except json.JSONDecodeError:
        print("Json inválido, aumentando o número de tokens permitidos e solicitando ao llm que envie novamente a resposta.\n")
        llm_parameters[2] += 1000
    else:
      raise ValueError(f"Não foi possivel obter um JSON válido para a classe {class_name} após várias tentativas.\n")

  # Gerar o relatório STRIDE em pdf para essa imagem
  image_name = image.split('.', 1)[0]
  pdf_path = gerar_relatorio_pdf(llm_respostas, f"/content/drive/MyDrive/Tech Challenge 5/reports/relatorio_stride_{image_name}.pdf")

Arquivo RAG 'file-storage-and-shared-filesystems.json' retornado com sucesso.
Resposta da llm para esta classe retornada com sucesso.

Arquivo RAG 'virtual-network-infrastructure.json' retornado com sucesso.
Resposta da llm para esta classe retornada com sucesso.

Arquivo RAG 'application-security-controls.json' retornado com sucesso.
Resposta da llm para esta classe retornada com sucesso.

Arquivo RAG 'enterprise-internal-systems.json' retornado com sucesso.
Resposta da llm para esta classe retornada com sucesso.

Arquivo RAG 'monitoring-and-telemetry.json' retornado com sucesso.
Resposta da llm para esta classe retornada com sucesso.

Arquivo RAG 'content-delivery-network.json' retornado com sucesso.
Resposta da llm para esta classe retornada com sucesso.

Arquivo RAG 'backup-and-disaster-recovery.json' retornado com sucesso.
Resposta da llm para esta classe retornada com sucesso.

Arquivo RAG 'secrets-and-key-management.json' retornado com sucesso.
Resposta da llm para esta classe r